## We will construct a linear model that can predict a car's mileage (mpg) by using its other attributes.

## Table of Contents

- [We will construct a linear model that can predict a car's mileage (mpg) by using its other attributes.](#we-will-construct-a-linear-model-that-can-predict-a-cars-mileage-mpg-by-using-its-other-attributes)
  - [Data Description:](#data-description)
- [Import Libraries](#import-libraries)
- [Load and explore the data](#load-and-explore-the-data)
- [Dealing with Missing Values](#dealing-with-missing-values)
- [Bivariate Analysis](#bivariate-analysis)
- [Create Dummy Variables](#create-dummy-variables)
- [Split Data](#split-data)
- [Fit Linear Model](#fit-linear-model)
  - [Interpretation of R-squared](#interpretation-of-r-squared)

### Data Description: 

The dataset has 9 variables, including the name of the car and its various attributes like horsepower, weight, region of origin, etc. Missing values in the data are marked by a series of question marks.

A detailed description of the variables is given below.

1. mpg: miles per gallon
2. cylinders: number of cylinders
3. displacement: engine displacement in cubic inches
4. horsepower: horsepower of the car
5. weight: weight of the car in pounds
6. acceleration: time taken, in seconds, to accelerate from O to 60 mph
7. model year: year of manufacture of the car (modulo 100)
8. origin: region of origin of the car (1 - American, 2 - European, 3 - Asian)
9. car name: name of the car

## Import Libraries


In [1]:
%pip install statsmodels pandas numpy nltk scikit-learn matplotlib VKPyKit  -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

# for visualizing data
import matplotlib.pyplot as plt
import seaborn as sns

# For randomized data splitting
from sklearn.model_selection import train_test_split

# To build linear regression_model
import statsmodels.api as sm

# To check model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error



In [3]:
from VKPyKit.DT import *
DT= DT()

**Note**: Model performance checking and the associated functions and libraries will be discussed in the video content of Week 2.

## Load and explore the data

**Note**: The code in the next cell will be used for loading data into Google Colab. If running it locally on Jupyter Notebook, this is not necessary. One can just use `cData = pd.read_csv("auto-mpg.csv")`

In [4]:
# from google.colab import files
# import io

# try:
#     uploaded
# except NameError:
#     uploaded = files.upload()

# cData = pd.read_csv(io.BytesIO(uploaded['auto-mpg.csv']))
cData = pd.read_csv("auto-mpg.csv")

In [5]:
# let's check the shape of the data
cData.shape

(398, 9)

In [25]:
# let's check the first 5 rows of the data
cData['horsepower'].replace('?', np.nan, inplace=True)
cData['horsepower'] = cData['horsepower'].astype(float)

/var/folders/sv/lfn41pw10vg_t2f_sttrywcw0000gn/T/ipykernel_72251/1562638542.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cData['horsepower'].replace('?', np.nan, inplace=True)


In [26]:
# let's check column types and number of values
cData.describe(include='all').T

,count,mean,std,min,25%,50%,75%,max
mpg,398.0,23.514573,7.815984,9.0,17.500,23.0,29.000,46.6
cylinders,398.0,5.454774,1.701004,3.0,4.000,4.0,8.000,8.0
displacement,398.0,193.425879,104.269838,68.0,104.250,148.5,262.000,455.0
horsepower,392.0,104.469388,38.491160,46.0,75.000,93.5,126.000,230.0
weight,398.0,2970.424623,846.841774,1613.0,2223.750,2803.5,3608.000,5140.0
acceleration,398.0,15.568090,2.757689,8.0,13.825,15.5,17.175,24.8
model year,398.0,76.010050,3.697627,70.0,73.000,76.0,79.000,82.0
origin,398.0,1.572864,0.802055,1.0,1.000,1.0,2.000,3.0


* Most of the columns in the data are numeric in nature ('int64' or 'float64' type).
* The horsepower and car name columns are string columns ('object' type).


We will be dropping the 'car name' column for prediction purposes.

In [8]:
cData = cData.drop(["car name"], axis=1)

## Dealing with Missing Values

In [18]:
# let's check the statistical summary of the data
cData.describe()

,mpg,cylinders,displacement,weight,acceleration,model year,origin
count,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000
mean,23.514573,5.454774,193.425879,2970.424623,15.568090,76.010050,1.572864
std,7.815984,1.701004,104.269838,846.841774,2.757689,3.697627,0.802055
min,9.000000,3.000000,68.000000,1613.000000,8.000000,70.000000,1.000000
25%,17.500000,4.000000,104.250000,2223.750000,13.825000,73.000000,1.000000
50%,23.000000,4.000000,148.500000,2803.500000,15.500000,76.000000,1.000000
75%,29.000000,8.000000,262.000000,3608.000000,17.175000,79.000000,2.000000
max,46.600000,8.000000,455.000000,5140.000000,24.800000,82.000000,3.000000


* The horsepower column is missing from the summary as it is not recognized as a numerical column.
* We will use the [`isdigit()`](https://python-reference.readthedocs.io/en/latest/docs/str/isdigit.html) function to check the values in the horsepower column that are not being recognized as numbers.
  - The `isdigit()` function is an inbuilt function of Python which is used to check whether all characters of a given string are digits or not
  - The function doesn't take any parameter and returns a *True* value only if all the characters of the string are digits, else it returns *False*
 


In [15]:
hpIsDigit = pd.DataFrame(
    cData.horsepower.str.isdigit()
)  # if the string is made of digits store True else False

# print the entries where isdigit = False
cData[hpIsDigit["horsepower"] == False]

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin


* We know that '?' denotes missing values.
* We will replace them with NaN. (this will help us deal with the missing values elegantly)

In [16]:
cData = cData.replace("?", np.nan)
cData[hpIsDigit["horsepower"] == False]

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin


* There are various ways to handle missing values.
* We can drop the rows, replace missing values with the mean/median of the available values, etc.
* Instead of dropping the rows, we will be replacing the missing values with median values.

In [27]:
# checking column medians
cData.median()

mpg               23.0
cylinders          4.0
displacement     148.5
horsepower        93.5
weight          2803.5
acceleration      15.5
model year        76.0
origin             1.0
dtype: float64

In [28]:
# Let's replace the missing values with median values of the columns.
# Note that we do not need to specify the column names below.
# Every column's missing value is replaced with that column's median respectively

medianFiller = lambda x: x.fillna(x.median())
cData = cData.apply(medianFiller, axis=0)

In [29]:
# let's convert the horsepower column from object type to float type
cData["horsepower"] = cData["horsepower"].astype(float)

**Let's replace the origin column values with their actual values.**

In [30]:
cData["origin"] = cData["origin"].replace({1: "america", 2: "europe", 3: "asia"})
cData.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin
0,18.0,8,307.0,130.0,3504,12.0,70,america
1,15.0,8,350.0,165.0,3693,11.5,70,america
2,18.0,8,318.0,150.0,3436,11.0,70,america
3,16.0,8,304.0,150.0,3433,12.0,70,america
4,17.0,8,302.0,140.0,3449,10.5,70,america


## Bivariate Analysis

A bivariate analysis among the different variables can be done using scatter matrix plot. Seaborn libs create a dashboard reflecting useful information about the dimensions. The result can be stored as a .png file.

In [44]:
# cData_attr = cData.iloc[:, 0:7]
# sns.pairplot(
#     cData_attr, diag_kind="kde"
# )  # to plot density curve instead of histogram on the diag

from VKPyKit.EDA import *
EDA = EDA()
EDA.pairplot_all(cData,features=None,hues=None,min_unique_values_for_pairplot=4, diag_kind="kde")  # to plot density curve instead of histogram on the diag

TypeError: 'NoneType' object is not iterable

<Figure size 1200x700 with 0 Axes>

* Observe that the relationship between 'mpg' and other attributes is not really linear.
* However, the plots also indicate that linearity would still capture quite a bit of useful information/pattern.
* Several assumptions of classical linear regression seem to be violated

## Create Dummy Variables


Values like 'america' cannot be read into an equation. Using substitutes like 1 for america, 2 for europe and 3 for asia would end up implying that European cars fall exactly half way between American and Asian cars! We don't want to impose such a baseless assumption!

So we create 3 simple true or false columns with titles equivalent to "Is this car American?", "Is this car European?" and "Is this car Asian?". These will be used as independent variables without imposing any kind of ordering between the three regions.

We will also be dropping one of those three columns to ensure there is no linear dependency between the three columns.

The pandas [`get_dummies()`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) function is used to convert a categorical variable to indicator/dummy variables (columns).

- It returns the dummy-coded data as a pandas dataframe
- In general, the `get_dummies()` function is applied to categorical columns in a pandas dataframe to generate dummy (one-hot encoded) columns

The `get_dummies()` function has the following parameters:

`pd.get_dummies(data, columns=['feature_name'], drop_first=True/False)`
 
where

- `data`: This is the first parameter of the function, and we pass the data that we want to create dummy indicator columns for
- `columns`: This parameter is used to pass the column names in the DataFrame to be encoded. If no value is passed, then the parameter defaults to *None*, in which case all the columns with *object* or *category* type will be converted into dummy variables
- `drop_first`: This parameter takes *True* or *False* as its values and is used to get *k-1* dummies out of *k* categorical levels (sorted in the ascending order of the alphabet) by removing the first level


In [ ]:
# drop_first=True will drop one of the three origin columns
cData = pd.get_dummies(cData, columns=["origin"], drop_first=True)
cData.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin_asia,origin_europe
0,18.0,8,307.0,130.0,3504,12.0,70,0,0
1,15.0,8,350.0,165.0,3693,11.5,70,0,0
2,18.0,8,318.0,150.0,3436,11.0,70,0,0
3,16.0,8,304.0,150.0,3433,12.0,70,0,0
4,17.0,8,302.0,140.0,3449,10.5,70,0,0


## Split Data

In [ ]:
# independent variables
X = cData.drop(["mpg"], axis=1)
# dependent variable
y = cData[["mpg"]]

In [ ]:
# let's add the intercept to data
X = sm.add_constant(X)

/usr/local/lib/python3.7/dist-packages/statsmodels/tsa/tsatools.py:117: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)


**We will now split X and y into train and test sets in a 70:30 ratio. We will use the `train_test_split()` function of sklearn to do the same.**

- sklearn, or Scikit-Learn, is a Python library that offers various features for data processing and modeling tasks
- In order to train a model properly, we need training and test datasets such that the model can be trained using the train data and can be tested on the unseen test data to get a better understandning of how the model is performing.
- If we have only one dataset provided, we'll need to split it into train and test sets by using the sklearn `train_test_split()` function

The sklearn [`test_train_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function has the following parameters:

`train_test_split(X, y, train_size, test_size, random_state)`

where 

- `X`, `y`: These are the first parameters of the function, and the set of independent (*X*) and dependent (*y*) variables from the dataset respectively have to be passed to them

- `train_size`: This parameter sets the size of the training dataset. There are three options to set this parameter:
  - *None*, which is the default
  - *int*, which requires the exact number of samples
  - *float*, which ranges from 0 to 1

- `test_size`: This parameter specifies the size of the testing dataset. It takes values similar to `train_size`
  - *int*, which requires the exact number of samples
  - *float*, which ranges from 0 to 1
  - If `train_size` is provided, this parameter sets the size of the test data to complement the training size
  - If the training size is set to default (i.e., *None*), it will be set to 0.25

- `random_state`:  This parameter controls the shuffling process. With the default value of *None*, we get the different train and test sets across different executions as the shuffling process is randomized. By passing a particular value, say 42, to this parameter, we get the same train and test sets across different executions

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1
)

In [ ]:
print(X_train.head())

     const  cylinders  displacement  horsepower  weight  acceleration  \
350    1.0          4         105.0        63.0    2215          14.9   
59     1.0          4          97.0        54.0    2254          23.5   
120    1.0          4         121.0       112.0    2868          15.5   
12     1.0          8         400.0       150.0    3761           9.5   
349    1.0          4          91.0        68.0    1985          16.0   

     model year  origin_asia  origin_europe  
350          81            0              0  
59           72            0              1  
120          73            0              1  
12           70            0              0  
349          81            1              0  


In [ ]:
print(X_test.head())

     const  cylinders  displacement  horsepower  weight  acceleration  \
174    1.0          6         171.0        97.0    2984          14.5   
359    1.0          4         141.0        80.0    3230          20.4   
250    1.0          8         318.0       140.0    3735          13.2   
274    1.0          5         131.0       103.0    2830          15.9   
283    1.0          6         232.0        90.0    3265          18.2   

     model year  origin_asia  origin_europe  
174          75            0              0  
359          81            0              1  
250          78            0              0  
274          78            0              1  
283          79            0              0  


## Fit Linear Model

We will use the [`OLS()`](https://www.statsmodels.org/dev/generated/statsmodels.regression.linear_model.OLS.html) function of the statsmodels library to fit the linear model.

- Statsmodels is a Python module that provides classes and functions for the estimation of many different statistical models, as well as for conducting statistical tests and statistical data exploration

- The `OLS()` function of the statsmodels.api module is used to perform OLS (Ordinary Least Squares) regression. It returns an OLS object

- The `fit()` method is called on this object for fitting the regression line to the data

- The `summary()` method is used to obtain a table which gives an extensive description about the regression results

In [ ]:
olsmod = sm.OLS(y_train, X_train)
olsres = olsmod.fit()

In [ ]:
# let's print the regression summary
print(olsres.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.814
Model:                            OLS   Adj. R-squared:                  0.809
Method:                 Least Squares   F-statistic:                     147.3
Date:                Thu, 28 Jul 2022   Prob (F-statistic):           1.20e-93
Time:                        14:36:10   Log-Likelihood:                -734.21
No. Observations:                 278   AIC:                             1486.
Df Residuals:                     269   BIC:                             1519.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           -21.2847      5.679     -3.748

### Interpretation of R-squared

* The R-squared value tells us that our model can explain 81.4% of the variance in the training set.